# NeuroFusionAI: Step 4 - Unified Inference Pipeline
This notebook loads the trained drawings classifier, voice classifier, telemonitoring regressor, and the multimodal fusion network, and executes a unified prediction.


In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
import glob
from PIL import Image
from torchvision import transforms

# Resolve project root (walk up until we find the 'datasets' directory)
PROJECT_ROOT = os.path.abspath('.')
for p in [os.path.abspath(x) for x in sys.path if x]:
    if os.path.isdir(os.path.join(p, 'datasets')):
        PROJECT_ROOT = p
        break
    elif os.path.isdir(os.path.join(os.path.dirname(p), 'datasets')):
        PROJECT_ROOT = os.path.dirname(p)
        break
while not os.path.isdir(os.path.join(PROJECT_ROOT, 'datasets')) and PROJECT_ROOT != os.path.dirname(PROJECT_ROOT):
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 1. Define Unified Prediction Function
We load all model checkpoints and construct a wrapper function that maps raw modalities to diagnosis status and severity scores.


In [ ]:
from models.image_model import ImageDrawingClassifier
from models.mri_model import MRIClassifier
from models.voice_model import VoiceMLPClassifier
from models.telemonitor_model import TelemonitorMLPRegressor
from models.fusion_model import MultimodalClassifier

# Load model definitions
img_model = ImageDrawingClassifier().to(device)
mri_model = MRIClassifier().to(device)
voice_model = VoiceMLPClassifier().to(device)
tele_model = TelemonitorMLPRegressor().to(device)
fusion_model = MultimodalClassifier(image_dim=256, mri_dim=256, voice_dim=22, clinical_dim=19, fusion_dim=32).to(device)

# Load checkpoints
if os.path.exists('outputs/checkpoints/image_best.pth'):
    img_model.load_state_dict(torch.load('outputs/checkpoints/image_best.pth', map_location=device))
if os.path.exists('outputs/checkpoints/mri_best.pth'):
    mri_model.load_state_dict(torch.load('outputs/checkpoints/mri_best.pth', map_location=device))
if os.path.exists('outputs/checkpoints/voice_mlp_best.pth'):
    voice_model.load_state_dict(torch.load('outputs/checkpoints/voice_mlp_best.pth', map_location=device))
if os.path.exists('outputs/checkpoints/telemonitor_mlp_best.pth'):
    tele_model.load_state_dict(torch.load('outputs/checkpoints/telemonitor_mlp_best.pth', map_location=device))
if os.path.exists('outputs/checkpoints/fusion_best.pth'):
    fusion_model.load_state_dict(torch.load('outputs/checkpoints/fusion_best.pth', map_location=device))

img_model.eval()
mri_model.eval()
voice_model.eval()
tele_model.eval()
fusion_model.eval()

def predict_patient(drawing_path, mri_path, voice_vector, tele_vector):
    # 1. Image Model
    img = Image.open(drawing_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    img_t = transform(img).unsqueeze(0).to(device)
    
    # 1.5 MRI Model
    mri_img = Image.open(mri_path).convert('RGB')
    mri_t = transform(mri_img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        img_logits = img_model(img_t)
        img_prob = torch.softmax(img_logits, dim=1)[0, 1].item()
        img_embedding = img_model.extract_features(img_t)
        
        mri_logits = mri_model(mri_t)
        mri_prob = torch.softmax(mri_logits, dim=1)[0, 1].item()
        mri_embedding = mri_model.extract_features(mri_t)
        
        # 2. Tabular Voice Model
        v_t = torch.tensor(voice_vector, dtype=torch.float32).unsqueeze(0).to(device)
        v_logits = voice_model(v_t)
        v_prob = torch.softmax(v_logits, dim=1)[0, 1].item()
        
        # 3. Telemonitoring Regressor Model
        t_t = torch.tensor(tele_vector, dtype=torch.float32).unsqueeze(0).to(device)
        tele_preds = tele_model(t_t).cpu().numpy()[0]
        motor_updrs, total_updrs = tele_preds[0], tele_preds[1]
        
        # 4. Multimodal Fusion Model
        fusion_logits = fusion_model(img_embedding, mri_embedding, v_t, t_t)
        fusion_prob = torch.softmax(fusion_logits, dim=1)[0, 1].item()
        
    print('================= NeuroFusionAI Results =================')
    print(f'Modality 1: Drawing Parkinson Likelihood:  {img_prob*100:.2f}%')
    print(f'Modality 1.5: MRI Parkinson Likelihood:    {mri_prob*100:.2f}%')
    print(f'Modality 2: Voice Parkinson Likelihood:     {v_prob*100:.2f}%')
    print(f'Multimodal Fused Parkinson Likelihood:      {fusion_prob*100:.2f}%')
    print(f'Severity Prediction (motor_UPDRS):          {motor_updrs:.2f}')
    print(f'Severity Prediction (total_UPDRS):          {total_updrs:.2f}')
    print('=========================================================')


## 2. Execute Prediction
Let's test our prediction pipeline on validation samples.


In [ ]:
sample_drawing = glob.glob('datasets/validation/images/parkinson/*.png')[0]
sample_mri = glob.glob('datasets/validation/mri/parkinson/*.png')[0]
sample_voice = pd.read_csv('datasets/validation/voice/oxford_validation.csv').drop(columns=['status']).iloc[0].values
sample_tele = pd.read_csv('datasets/validation/telemonitoring/telemonitor_validation.csv').drop(columns=['motor_UPDRS', 'total_UPDRS']).iloc[0].values

predict_patient(sample_drawing, sample_mri, sample_voice, sample_tele)
